In [0]:
# """
# ================================================================================
# Superstore Gold Fact ETL Orchestrator
# ================================================================================
# Purpose       : Master orchestrator for building and maintaining all fact 
#                 tables in the Gold layer. Fully embeds the fact ETL logic 
#                 instead of calling an external function.

# Design Notes  :
# 1. Supports multiple fact tables configured centrally via `fact_table_configs`.
# 2. Implements full ETL orchestration with observability using master_run_id and 
#    layer_run_id.
# 3. Optimizations include Z-Ordering for Gold tables and Delta VACUUM for stale 
#    files.
# 4. Structured logging enables easy monitoring and dashboarding for production 
#    pipelines.

# Usage         :
#     python run_gold_fact_pipelines.py  # Execute all configured fact pipelines
# ================================================================================
# """


from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

import os, sys, time, yaml, importlib
from datetime import datetime
import uuid

# -----------------------------
# Fetch environment from master task
# -----------------------------
env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_gold/core")

# -----------------------------
# Import platform helpers and logging
# -----------------------------
from superstore_logger import get_superstore_logger, log_event
from superstore_orchestrator_execution_context import get_execution_context
from superstore_platform_constants import GOLD_LAYER
from superstore_platform_config import table, get_silver_schema, get_gold_schema, get_metrics_schema

# -----------------------------
# Get execution context
# -----------------------------
execution_context = get_execution_context(dbutils, allow_standalone=True)
master_run_id = execution_context["master_run_id"]
layer_run_id = execution_context["layer_run_id"]

# -----------------------------
# Logger initialization
# -----------------------------
logger_gold_facts_orchestrator = get_superstore_logger("superstore_gold_facts_orchestrator")

log_event(
    logger_gold_facts_orchestrator,
    "INFO",
    "Initiating Superstore Gold layer ETL facts orchestrator",
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=GOLD_LAYER
)

# -----------------------------
# Load fact table configuration
# -----------------------------
config_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/configs/superstore_gold_config/superstore_gold_facts_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
fact_table_configs = config["fact_table_configs"]

# -----------------------------
# Import Gold framework modules
# -----------------------------
import superstore_gold_facts_framework
importlib.reload(superstore_gold_facts_framework)
from superstore_gold_facts_framework import (
    read_silver_table,
    prepare_fact_columns,
    create_gold_table_if_not_exists,
    merge_fact_into_gold,
    collect_fact_metrics,
    optimize_gold_table,
    vacuum_gold_table,
)

# -----------------------------
# Master Orchestrator Function
# -----------------------------
def run_all_fact_pipelines(spark, master_run_id: str):
    log_event(
        logger_gold_facts_orchestrator,
        "INFO",
        "Starting Gold fact ETL for all configured tables",
        total_tables=len(fact_table_configs),
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=GOLD_LAYER
    )

    for table_name, cfg in fact_table_configs.items():

        # -----------------------------
        # Dynamically resolve full catalog.schema.table
        # -----------------------------
        silver_tbl = table(get_silver_schema(), cfg["silver_tbl"])
        gold_tbl = table(get_gold_schema(), cfg["gold_tbl"])
        metrics_table = table(get_metrics_schema(), cfg["metrics_table"])

        layer_name = cfg["layer_name"]
        table_type = cfg["table_type"]
        required_columns = cfg["required_columns"]
        natural_keys = cfg["natural_keys"]
        hash_column = cfg["hash_column"]
        z_order_cols = cfg.get("z_order_cols", [])

        start_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
        start_time = time.time()
        run_status = "SUCCESS"
        notes = ""
        has_incremental_rows = False

        try:
            log_event(
                logger_gold_facts_orchestrator,
                "INFO",
                f"Starting fact pipeline for table: {gold_tbl}",
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 1: Read Silver table
            df_silver = read_silver_table(
                spark,
                silver_table=silver_tbl,
                gold_table=gold_tbl,
                required_columns=required_columns,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            has_incremental_rows = df_silver.limit(1).count()

            if not has_incremental_rows:
                log_event(
                    logger_gold_facts_orchestrator,
                    "INFO",
                    f"No new rows/files for {gold_tbl}; skipping merge",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )
                collect_fact_metrics(
                    spark,
                    df=df_silver,
                    metrics_table=metrics_table,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    table_name=gold_tbl,
                    table_type=table_type,
                    start_ts=start_ts,
                    end_ts=start_ts,
                    duration_secs=0,
                    run_status="skipped",
                    notes="No new rows/files to process",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )
                continue

            # Step 2: Prepare fact columns
            df_prepared = prepare_fact_columns(
                df_silver,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 3: Create Gold table if missing
            create_gold_table_if_not_exists(
                df_prepared,
                gold_tbl,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 4: Merge fact into Gold
            merge_fact_into_gold(
                df_prepared,
                gold_tbl,
                natural_keys,
                hash_column,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

        except Exception as e:
            run_status = "FAILED"
            notes = str(e)
            logger_gold_facts_orchestrator.error(f"Fact pipeline failed for {gold_tbl}: {e}")
            raise

        finally:
            if has_incremental_rows:
                end_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
                duration_secs = int(time.time() - start_time)

                # Step 5: Metrics
                collect_fact_metrics(
                    spark,
                    df=df_silver,
                    metrics_table=metrics_table,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    table_name=gold_tbl,
                    table_type=table_type,
                    start_ts=start_ts,
                    end_ts=end_ts,
                    duration_secs=duration_secs,
                    run_status=run_status,
                    notes=notes,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                # Step 6: Optimize Gold
                optimize_gold_table(
                    spark,
                    gold_tbl,
                    z_order_cols,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                # Step 7: Vacuum Gold
                vacuum_gold_table(
                    spark,
                    gold_tbl,
                    retention_hours=168,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                log_event(
                    logger_gold_facts_orchestrator,
                    "INFO",
                    f"Completed fact pipeline for table: {gold_tbl}",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

    log_event(
        logger_gold_facts_orchestrator,
        "INFO",
        "Completed executing fact pipeline for all tables",
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=GOLD_LAYER
    )


# -------------------------------
# Execute
# -------------------------------
if __name__ == "__main__":
    run_all_fact_pipelines(spark, master_run_id)
# dbutils.notebook.exit(layer_run_id)
